# AEGIS Paper Figures

This notebook reads a SC26 evaluation campaign from `results/`, computes paper-facing summaries, and regenerates figure assets for the manuscript.

## Expected Inputs

A complete campaign directory should contain:
- `baseline_comparison.md`
- `bpf_microbenchmark_*.json`
- `real_latency_*.json`
- `real_latency_sweep.json`
- `real_ablation.json`
- `simulated_all_attacks.json`
- `simulated_ablation.json`
- `simulated_false_positive.json`
- `simulated_performance.json`

## Result Selection

If `AEGIS_RESULTS_DIR` is set, the notebook uses that directory.
Otherwise it selects the newest `results/sc26_run_*` directory automatically.

## False-Positive Study Note

`simulated_false_positive.json` now represents an expanded benign suite rather than only a handful of clean workflows. The suite includes:
- standard single-agent HPC workflows
- authorized multi-agent collaboration over project and `/tmp` paths
- budget-edge but allowed LLM reporting
- tool-heavy benign postprocessing

Interpret `false_positive_summary.png` as a benign-suite coverage figure with an explicit no-false-positive outcome panel.

In [1]:
import json
import os
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap

plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.16,
    'axes.axisbelow': True,
})

if Path.cwd().name == 'figures':
    ROOT = Path.cwd().resolve().parent
else:
    ROOT = Path.cwd().resolve()

FIG_DIR = ROOT / 'figures'
RESULTS_ROOT = ROOT / 'results'
RESULT_DIR_HINT = os.environ.get('AEGIS_RESULTS_DIR', '').strip()


def resolve_result_dir() -> Path:
    if RESULT_DIR_HINT:
        candidate = Path(RESULT_DIR_HINT)
        if not candidate.is_absolute():
            candidate = (ROOT / candidate).resolve()
        if candidate.exists():
            return candidate
        raise FileNotFoundError(f'AEGIS_RESULTS_DIR does not exist: {candidate}')
    candidates = sorted([p for p in RESULTS_ROOT.glob('sc26_run_*') if p.is_dir()], key=lambda p: p.name)
    if not candidates:
        raise FileNotFoundError('No results/sc26_run_* directories found')
    return candidates[-1]


RESULT_DIR = resolve_result_dir()
print(f'Using results directory: {RESULT_DIR.relative_to(ROOT)}')

Using results directory: results/sc26_run_20260331T172639Z


In [2]:
ATTACK_ORDER = ['FS Inj', 'CoLoc', 'Supply', 'CordEx']
ATTACK_KEY_ORDER = ['filesystem', 'colocation', 'supply_chain', 'coordinated']
ATTACK_LABELS = {
    'filesystem': 'Filesystem',
    'colocation': 'Co-Location',
    'supply_chain': 'Supply Chain',
    'coordinated': 'Coordinated',
}


def save_fig(fig, filename: str) -> Path:
    path = FIG_DIR / filename
    fig.savefig(path, bbox_inches='tight')
    print(f'Saved {path.relative_to(ROOT)}')
    return path


def load_json(filename: str):
    path = RESULT_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f'Missing artifact: {path}')
    return json.loads(path.read_text())


def parse_baseline_table(path: Path):
    rows = []
    in_table = False
    for line in path.read_text().splitlines():
        stripped = line.strip()
        if stripped.startswith('| Defense |'):
            in_table = True
            continue
        if in_table and stripped.startswith('|---------'):
            continue
        if in_table:
            if not stripped.startswith('|'):
                break
            cols = [c.strip() for c in stripped.strip('|').split('|')]
            if len(cols) != 7:
                continue
            rows.append({
                'defense': cols[0],
                'detections': [1 if 'DET' in c else 0 for c in cols[1:5]],
                'rate': float(cols[5].replace('%', '').strip()),
                'avg_time_ms': float(cols[6].split('±')[0].strip()),
            })
    if not rows:
        raise ValueError(f'Could not parse baseline comparison table: {path}')
    return rows


def collect_real_latency_summaries():
    items = []
    for path in sorted(RESULT_DIR.glob('real_latency_*.json')):
        if path.name == 'real_latency_sweep.json':
            continue
        payload = json.loads(path.read_text())
        items.append(payload['summary'])
    items.sort(key=lambda item: ATTACK_KEY_ORDER.index(item['attack']))
    return items


def collect_microbenchmarks():
    items = []
    for path in sorted(RESULT_DIR.glob('bpf_microbenchmark_*.json')):
        payload = json.loads(path.read_text())
        items.append({
            'mode': payload['config']['mode'],
            'elapsed_pct': payload['overhead']['elapsed_pct'],
            'task_clock_pct': payload['overhead']['task_clock_pct'],
            'ops_per_sec_pct': abs(payload['overhead']['ops_per_sec_pct']),
        })
    mode_order = {'openat': 0, 'read': 1, 'connect': 2, 'execve': 3}
    items.sort(key=lambda row: mode_order.get(row['mode'], 99))
    return items


def load_simulated_ablation_matrix(payload):
    config_names = list(payload['results'].keys())
    attack_names = list(next(iter(payload['results'].values()))['results'].keys())
    matrix = np.zeros((len(config_names), len(attack_names)))
    rates = []
    for i, cfg in enumerate(config_names):
        rates.append(payload['results'][cfg]['detection_rate'])
        for j, attack in enumerate(attack_names):
            matrix[i, j] = 100.0 if payload['results'][cfg]['results'][attack]['detected'] else 0.0
    return config_names, attack_names, matrix, rates


baseline_rows = parse_baseline_table(RESULT_DIR / 'baseline_comparison.md')
real_latency = collect_real_latency_summaries()
real_latency_sweep = load_json('real_latency_sweep.json')
real_ablation = load_json('real_ablation.json')
simulated_all = load_json('simulated_all_attacks.json')
simulated_ablation = load_json('simulated_ablation.json')
simulated_fp = load_json('simulated_false_positive.json')
simulated_perf = load_json('simulated_performance.json')
microbench = collect_microbenchmarks()

print('Artifacts loaded:')
print(f'  baseline rows: {len(baseline_rows)}')
print(f'  real attacks: {len(real_latency)}')
print(f'  microbenchmark modes: {len(microbench)}')
print(f'  simulated attack coverage entries: {len(simulated_all["results"])}')

Artifacts loaded:
  baseline rows: 5
  real attacks: 4
  microbenchmark modes: 3
  simulated attack coverage entries: 4


In [3]:
summary_rows = []
for result in simulated_all['results']:
    summary_rows.append({
        'attack': result['experiment_name'],
        'detected': result['detection_success'],
        'detections': result['num_detections'],
        'detection_time_ms': result['detection_time_ms'],
        'exfiltrated_bytes': result['exfiltrated_bytes'],
    })
print('Simulated attack summary:')
for row in summary_rows:
    print(f"  {row['attack']}: detected={row['detected']} detections={row['detections']} exfil={row['exfiltrated_bytes']}B time={row['detection_time_ms']:.4f}ms")

Simulated attack summary:
  Filesystem-Mediated Injection: detected=True detections=1 exfil=68B time=0.3355ms
  Multi-User Co-Location Injection: detected=True detections=1 exfil=50B time=0.0474ms
  Supply Chain Injection via Skills: detected=True detections=7 exfil=519B time=0.1233ms
  Coordinated Multi-Agent Exfiltration: detected=True detections=7 exfil=521B time=0.1292ms


In [4]:
# Figure 1: baseline comparison.

defenses = [row['defense'] for row in baseline_rows]
mat = np.array([row['detections'] for row in baseline_rows], dtype=float)
rates = [row['rate'] for row in baseline_rows]
analysis_times = [row['avg_time_ms'] for row in baseline_rows]

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(12.6, 4.9), gridspec_kw={'width_ratios': [4.6, 1.8]}, constrained_layout=True)
cmap = ListedColormap(['#f4a6a6', '#8ecf9f'])
ax0.imshow(mat, cmap=cmap, vmin=0, vmax=1, aspect='auto')
ax0.set_xticks(range(len(ATTACK_ORDER)))
ax0.set_xticklabels(ATTACK_ORDER)
ax0.set_yticks(range(len(defenses)))
ax0.set_yticklabels(defenses)
ax0.set_title('Detection Outcomes by Defense')
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        ax0.text(j, i, 'DET' if mat[i, j] else 'MISS', ha='center', va='center', fontsize=10, fontweight='bold')
for tick in ax0.get_yticklabels():
    if tick.get_text() == 'AEGIS (Ours)':
        tick.set_fontweight('bold')

bars = ax1.barh(defenses, analysis_times, color=['#9bb8d3' if d != 'AEGIS (Ours)' else '#d95f02' for d in defenses])
ax1.set_title('Mean Analysis Time')
ax1.set_xlabel('ms')
ax1.invert_yaxis()
for bar, rate in zip(bars, rates):
    ax1.text(bar.get_width(), bar.get_y() + bar.get_height()/2, f' {rate:.0f}%', va='center', fontsize=9)

save_fig(fig, 'baseline_comparison.png')
plt.show()

Saved figures/baseline_comparison.png


In [5]:
# Figure 2: measured per-attack outcomes on the real path.

attack_names = [ATTACK_LABELS[item['attack']] for item in real_latency]
latencies = [
    item['median_detection_latency_ms'] if item['median_detection_latency_ms'] is not None else np.nan
    for item in real_latency
]
exfil_kb = [
    (item['median_exfiltrated_bytes'] / 1024.0) if item['median_exfiltrated_bytes'] is not None else np.nan
    for item in real_latency
]
cpu_overhead = [item['median_cpu_overhead_percent'] for item in real_latency]
detected_text = [f"{item['detected_trials']}/{item['repeats']}" for item in real_latency]

fig, axes = plt.subplots(1, 3, figsize=(13.6, 4.4), constrained_layout=True)
colors = ['#4e79a7', '#59a14f', '#f28e2b', '#e15759']

bars0 = axes[0].bar(attack_names, np.nan_to_num(latencies, nan=0.0), color=colors)
axes[0].set_title('Measured Detection Latency')
axes[0].set_ylabel('ms')
axes[0].tick_params(axis='x', rotation=20)
for bar, value in zip(bars0, latencies):
    label = 'MISS' if np.isnan(value) else f'{value:.0f}'
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 8, label, ha='center', va='bottom', fontsize=9)

bars1 = axes[1].bar(attack_names, np.nan_to_num(exfil_kb, nan=0.0), color=colors)
axes[1].set_title('Exfiltration Before Detection')
axes[1].set_ylabel('KB')
axes[1].tick_params(axis='x', rotation=20)
for bar, value in zip(bars1, exfil_kb):
    label = 'N/A' if np.isnan(value) else f'{value:.2f}'
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, label, ha='center', va='bottom', fontsize=9)

bars2 = axes[2].bar(attack_names, cpu_overhead, color=colors)
axes[2].set_title('Verifier CPU Overhead')
axes[2].set_ylabel('%')
axes[2].tick_params(axis='x', rotation=20)
for bar, text in zip(bars2, detected_text):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, text, ha='center', va='bottom', fontsize=9)

save_fig(fig, 'attack_results.png')
plt.show()

Saved figures/attack_results.png


In [6]:
# Figure 3: real ablation heatmap.

config_keys = real_ablation['configs']
attack_keys = real_ablation['attacks']
summary = real_ablation['summary']
lookup = {(row['config_key'], row['attack_key']): row for row in summary}
config_names = [next(row['config_name'] for row in summary if row['config_key'] == key) for key in config_keys]
attack_names = [next(row['attack_name'] for row in summary if row['attack_key'] == key) for key in attack_keys]
rate_matrix = np.zeros((len(config_keys), len(attack_keys)))
annotations = [['' for _ in attack_keys] for _ in config_keys]

for i, cfg_key in enumerate(config_keys):
    for j, attack_key in enumerate(attack_keys):
        row = lookup[(cfg_key, attack_key)]
        rate = 100.0 * row['detected_trials'] / max(row['total_trials'], 1)
        rate_matrix[i, j] = rate
        exfil = row['median_exfiltrated_bytes']
        if exfil is None:
            annotations[i][j] = f"{rate:.0f}%\nN/A"
        else:
            annotations[i][j] = f"{rate:.0f}%\n{int(exfil)} B"

fig, ax = plt.subplots(figsize=(10.8, 5.2), constrained_layout=True)
im = ax.imshow(rate_matrix, cmap='YlGnBu', vmin=0, vmax=100, aspect='auto')
ax.set_xticks(range(len(attack_names)))
ax.set_xticklabels(attack_names, rotation=20, ha='right')
ax.set_yticks(range(len(config_names)))
ax.set_yticklabels(config_names)
ax.set_title('Real Ablation: Detection Rate and Exfiltration')
for i in range(len(config_names)):
    for j in range(len(attack_names)):
        ax.text(j, i, annotations[i][j], ha='center', va='center', fontsize=9, fontweight='bold')
fig.colorbar(im, ax=ax, label='Detection rate (%)')

save_fig(fig, 'ablation_heatmap.png')
plt.show()

Saved figures/ablation_heatmap.png


In [7]:
# Figure 4: latency tradeoff across attestation intervals.

sweep_summary = sorted(real_latency_sweep['summary'], key=lambda row: row['interval'])
intervals = [row['interval'] for row in sweep_summary]
avg_latency = [row['avg_latency_ms'] for row in sweep_summary]
exfil_kb = [row['total_exfil_kb'] for row in sweep_summary]
cpu_pct = [row['cpu_overhead'] for row in sweep_summary]

fig, ax = plt.subplots(figsize=(8.8, 5.0), constrained_layout=True)
ax.plot(intervals, avg_latency, marker='o', linewidth=2.4, color='#d95f02', label='Detection latency (ms)')
ax.set_xscale('log')
ax.set_xlabel('Attestation interval (s)')
ax.set_ylabel('Detection latency (ms)', color='#d95f02')
ax.tick_params(axis='y', labelcolor='#d95f02')
ax.set_title('Measured Detection Latency Tradeoff')

ax2 = ax.twinx()
ax2.plot(intervals, exfil_kb, marker='s', linestyle='--', linewidth=2.0, color='#4e79a7', label='Exfiltration (KB)')
ax2.plot(intervals, cpu_pct, marker='^', linestyle=':', linewidth=2.0, color='#1b9e77', label='CPU overhead (%)')
ax2.set_ylabel('Exfiltration (KB) / CPU overhead (%)')

lines = ax.get_lines() + ax2.get_lines()
labels = [line.get_label() for line in lines]
ax.legend(lines, labels, loc='upper left', frameon=False)

save_fig(fig, 'latency_tradeoff.png')
plt.show()

Saved figures/latency_tradeoff.png


In [8]:
# Figure 5: direct probe microbenchmark breakdown.

modes = [row['mode'] for row in microbench]
elapsed_pct = [row['elapsed_pct'] for row in microbench]
task_clock_pct = [row['task_clock_pct'] for row in microbench]
ops_delta_pct = [row['ops_per_sec_pct'] for row in microbench]

fig, ax = plt.subplots(figsize=(9.2, 4.8), constrained_layout=True)
x = np.arange(len(modes))
width = 0.24
ax.bar(x - width, elapsed_pct, width=width, color='#4e79a7', label='Elapsed overhead')
ax.bar(x, task_clock_pct, width=width, color='#59a14f', label='Task-clock overhead')
ax.bar(x + width, ops_delta_pct, width=width, color='#f28e2b', label='Ops/sec delta')
ax.set_xticks(x)
ax.set_xticklabels(modes)
ax.set_ylabel('%')
ax.set_title('Direct Probe Overhead by Syscall Family')
ax.legend(frameon=False, ncol=3)

save_fig(fig, 'microbenchmark_breakdown.png')
plt.show()

Saved figures/microbenchmark_breakdown.png


In [9]:
# Figure 6: simulated ablation breakdown.

config_names, attack_names, sim_ablation_matrix, sim_rates = load_simulated_ablation_matrix(simulated_ablation)
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(12.4, 5.0), gridspec_kw={'width_ratios': [4.2, 1.8]}, constrained_layout=True)
im = ax0.imshow(sim_ablation_matrix, cmap='PuBuGn', vmin=0, vmax=100, aspect='auto')
ax0.set_xticks(range(len(attack_names)))
ax0.set_xticklabels(attack_names, rotation=20, ha='right')
ax0.set_yticks(range(len(config_names)))
ax0.set_yticklabels(config_names)
ax0.set_title('Simulated Ablation by Attack Class')
for i in range(len(config_names)):
    for j in range(len(attack_names)):
        ax0.text(j, i, f"{sim_ablation_matrix[i, j]:.0f}%", ha='center', va='center', fontsize=9, fontweight='bold')
fig.colorbar(im, ax=ax0, label='Detection (%)')

bars = ax1.barh(config_names, sim_rates, color=['#d95f02' if name == 'Full AEGIS' else '#9bb8d3' for name in config_names])
ax1.set_title('Overall Detection Rate')
ax1.set_xlabel('%')
ax1.invert_yaxis()
for bar, rate in zip(bars, sim_rates):
    ax1.text(bar.get_width(), bar.get_y() + bar.get_height()/2, f' {rate:.0f}%', va='center', fontsize=9)

save_fig(fig, 'simulated_ablation_breakdown.png')
plt.show()

Saved figures/simulated_ablation_breakdown.png


In [10]:
# Figure 7: scaling and workload sweeps from the simulated performance study.

interval_sweep = sorted(simulated_perf['interval_sweep'], key=lambda row: row['attestation_interval'])
agent_sweep = sorted(simulated_perf['agent_count_sweep'], key=lambda row: row['agent_count'])
workload_sweep = simulated_perf['workload_type_sweep']

fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.6), constrained_layout=True)

axes[0].plot([row['attestation_interval'] for row in interval_sweep], [row['overhead_percent'] for row in interval_sweep], marker='o', color='#4e79a7')
axes[0].set_xscale('log')
axes[0].set_title('Overhead vs Interval')
axes[0].set_xlabel('Attestation interval (s)')
axes[0].set_ylabel('Overhead (%)')

axes[1].plot([row['agent_count'] for row in agent_sweep], [row['overhead_percent'] for row in agent_sweep], marker='o', color='#e15759')
axes[1].set_title('Overhead vs Agent Count')
axes[1].set_xlabel('Agent count')
axes[1].set_ylabel('Overhead (%)')

axes[2].bar([row['workload_type'] for row in workload_sweep], [row['overhead_percent'] for row in workload_sweep], color='#76b7b2')
axes[2].set_title('Overhead by Workload')
axes[2].set_ylabel('Overhead (%)')
axes[2].tick_params(axis='x', rotation=20)

save_fig(fig, 'scaling_sweep.png')
plt.show()

Saved figures/scaling_sweep.png


In [11]:
# Figure 8: benign-suite coverage and false-positive outcome.

workflow_names = list(simulated_fp['workflows'].keys())
workflow_rows = [simulated_fp['workflows'][name] for name in workflow_names]
actions = [row['actions_count'] for row in workflow_rows]
logged_actions = [row.get('logged_actions', row['actions_count']) for row in workflow_rows]
agent_counts = [row.get('agent_count', 1) for row in workflow_rows]
detections = [row['detections'] for row in workflow_rows]
total_actions = simulated_fp.get('total_actions', sum(actions))
workflow_count = simulated_fp.get('workflow_count', len(workflow_names))

labels = [f"{name} ({agents} agent{'s' if agents != 1 else ''})" for name, agents in zip(workflow_names, agent_counts)]
y = np.arange(len(labels))

fig, (ax0, ax1) = plt.subplots(
    1,
    2,
    figsize=(13.2, 5.4),
    gridspec_kw={'width_ratios': [4.8, 1.6]},
    constrained_layout=True,
)

bar_h = 0.36
ax0.barh(y + bar_h / 2, actions, height=bar_h, color='#4e79a7', label='Workflow actions')
ax0.barh(y - bar_h / 2, logged_actions, height=bar_h, color='#9ecae1', label='Logged events')
ax0.set_yticks(y)
ax0.set_yticklabels(labels)
ax0.invert_yaxis()
ax0.set_xlabel('Count')
ax0.set_title('Expanded Benign Suite Coverage')
ax0.legend(frameon=False, loc='lower right')
for yi, agent_count, action_count, logged_count in zip(y, agent_counts, actions, logged_actions):
    ax0.text(max(action_count, logged_count) + 0.15, yi, f"{agent_count} agent" + ('s' if agent_count != 1 else ''), va='center', fontsize=9)

ax1.set_title('False-Positive Outcome')
ax1.set_xlim(0, 1)
ax1.set_ylim(-0.5, len(labels) - 0.5)
ax1.set_xticks([])
ax1.set_yticks([])
for yi, det in zip(y, detections):
    color = '#2ca02c' if det == 0 else '#d62728'
    marker = '✓' if det == 0 else str(det)
    ax1.scatter([0.5], [yi], s=1600, color=color, alpha=0.18)
    ax1.text(0.5, yi, marker, ha='center', va='center', fontsize=18, fontweight='bold', color=color)

ax1.text(0.5, len(labels) - 0.1, f"Overall FP rate\\n{simulated_fp['overall_fp_rate']:.1f}%", ha='center', va='top', fontsize=11, fontweight='bold')
ax1.text(0.5, len(labels) - 1.1, f"{total_actions} actions across\\n{workflow_count} workflows", ha='center', va='top', fontsize=10)

save_fig(fig, 'false_positive_summary.png')
plt.show()


Saved figures/false_positive_summary.png


In [13]:
# Figure 9: combined measured-overhead summary used in the paper draft.

sweep_summary = sorted(real_latency_sweep['summary'], key=lambda row: row['interval'])
intervals = [row['interval'] for row in sweep_summary]
avg_latency = [row['avg_latency_ms'] for row in sweep_summary]
cpu_pct = [row['cpu_overhead'] for row in sweep_summary]

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(12.8, 4.8), constrained_layout=True)
mode_names = [row['mode'] for row in microbench]
elapsed_pct = [row['elapsed_pct'] for row in microbench]
ops_delta_pct = [row['ops_per_sec_pct'] for row in microbench]
bar_x = np.arange(len(mode_names))
width = 0.36
ax0.bar(bar_x - width/2, elapsed_pct, width=width, label='Elapsed overhead', color='#4e79a7')
ax0.bar(bar_x + width/2, ops_delta_pct, width=width, label='Ops/sec delta', color='#f28e2b')
ax0.set_xticks(bar_x)
ax0.set_xticklabels(mode_names)
ax0.set_ylabel('%')
ax0.set_title('Direct Probe Overhead')
ax0.legend(frameon=False)

ax1.plot(intervals, avg_latency, marker='o', linewidth=2.2, color='#d95f02', label='Detection latency (ms)')
ax1.set_xscale('log')
ax1.set_xlabel('Attestation interval (s)')
ax1.set_ylabel('Detection latency (ms)', color='#d95f02')
ax1.tick_params(axis='y', labelcolor='#d95f02')
ax1.set_title('Latency vs Attestation Interval')
ax1b = ax1.twinx()
ax1b.plot(intervals, cpu_pct, marker='s', linestyle='--', color='#1b9e77', label='CPU overhead (%)')
ax1b.set_ylabel('CPU overhead (%)', color='#1b9e77')
ax1b.tick_params(axis='y', labelcolor='#1b9e77')

save_fig(fig, 'performance_overhead.png')
plt.show()

Saved figures/performance_overhead.png


In [14]:
generated = [
    'baseline_comparison.png',
    'attack_results.png',
    'ablation_heatmap.png',
    'latency_tradeoff.png',
    'microbenchmark_breakdown.png',
    'simulated_ablation_breakdown.png',
    'scaling_sweep.png',
    'false_positive_summary.png',
    'performance_overhead.png',
]
for name in generated:
    path = FIG_DIR / name
    print(f'{name}: {path.stat().st_size / 1024:.1f} KB')

baseline_comparison.png: 187.8 KB
attack_results.png: 193.5 KB
ablation_heatmap.png: 292.4 KB
latency_tradeoff.png: 209.9 KB
microbenchmark_breakdown.png: 89.3 KB
simulated_ablation_breakdown.png: 300.2 KB
scaling_sweep.png: 257.5 KB
false_positive_summary.png: 291.6 KB
detection_radar.png: 215.1 KB
performance_overhead.png: 223.4 KB
